### Calculating everything needed as tarHMM input
- all pretrained SAM3 tracks on the gt crops; t cell stats only
- Use AnalysisEnv

In [ ]:
import pickle
import numpy as np

import zipfile
from io import BytesIO
import tifffile

import os

from pathlib import Path
import yaml

os.chdir("/gladstone/engelhardt/lab/jutran/lci/MarsonImagingPipeline")
from scripts.utils.StatUtils import *
from scripts.utils.CellTypingUtils import *

In [ ]:
cvat_base_dir = '/gladstone/engelhardt/lab/MarsonLabIncucyteData/UltrackAnalysis/groundTruthTracks/TCR-T/'
SAM3_base_dir = "/gladstone/engelhardt/lab/jutran/lci/MarsonImagingPipeline/data/cell_type_assignment/original_SAM3_track_assignments/"

crop_ids = ['B4_t50t100y200y350x750x900',
            'B8_t50t100y200y350x750x900',
            'E4_t50t100y200y350x750x900',
            'B4_t250t300y200y350x750x900',
            'B8_t250t300y200y350x750x900',
            'E4_t250t300y200y350x750x900']

### Calculate masking metadata

In [3]:
sam3_tracks_per_well = {}
t_cell_tracks_per_well = {}
cancer_tracks_per_well = {}

for crop in crop_ids:
    well_id = crop.split('_')[0]

    # load all tracks 
    sam3_tracks = tifffile.imread(os.path.join(SAM3_base_dir, crop, f'reindexed_tracks.tiff'))
    sam3_tracks_per_well[crop] = sam3_tracks

    cell_type_dict = pickle.load(open(os.path.join(SAM3_base_dir, crop, "full_cell_type_dict.pkl"), "rb"))
    t_cell_tracks_per_well[crop] = filter_tracks("t_cell", sam3_tracks, cell_type_dict)
    cancer_tracks_per_well[crop] = filter_tracks("cancer", sam3_tracks, cell_type_dict)


In [10]:
# obtain max num_cells first
max_num_cells = 0
for crop in crop_ids:
    t_cell_tracks = t_cell_tracks_per_well[crop]
    T = t_cell_tracks.shape[0]
    all_cell_ids = np.unique(t_cell_tracks[t_cell_tracks > 0])
    num_cells = len(all_cell_ids)
    print(f"crop={crop}, T={T}, num_cells={num_cells}")
    if num_cells > max_num_cells:
        max_num_cells = num_cells
    print(f"max_num_cells so far: {max_num_cells}")

crop=B4_t50t100y200y350x750x900, T=50, num_cells=57
max_num_cells so far: 57
crop=B8_t50t100y200y350x750x900, T=50, num_cells=77
max_num_cells so far: 77
crop=E4_t50t100y200y350x750x900, T=50, num_cells=22
max_num_cells so far: 77
crop=B4_t250t300y200y350x750x900, T=50, num_cells=55
max_num_cells so far: 77
crop=B8_t250t300y200y350x750x900, T=50, num_cells=88
max_num_cells so far: 88
crop=E4_t250t300y200y350x750x900, T=50, num_cells=27
max_num_cells so far: 88


In [ ]:
# calculate actual statistics
data = {}

# Initialize all masks
active_mask = np.zeros([50,0], dtype=bool)
is_division_mask = np.zeros([50,0], dtype=bool)
is_new_root_mask = np.zeros([50,0], dtype=bool)
parent_indices = np.zeros([50,0], dtype=np.int32)

active_mask_list = []
is_division_mask_list = []
is_new_root_mask_list = []
parent_indices_list = []

for crop in crop_ids:
    t_cell_tracks = t_cell_tracks_per_well[crop]

    T = t_cell_tracks.shape[0]
    all_cell_ids = np.unique(t_cell_tracks[t_cell_tracks > 0])
    all_cell_ids.sort()
    num_cells = len(all_cell_ids)
    id_to_col = {int(cid): i for i, cid in enumerate(all_cell_ids)}

    print(f"crop={crop}, T={T}, num_cells={num_cells}")

    # Initialize all masks
    crop_active_mask = np.zeros((T, max_num_cells), dtype=bool)
    crop_is_division_mask = np.zeros((T, max_num_cells), dtype=bool)
    crop_is_new_root_mask = np.zeros((T, max_num_cells), dtype=bool)
    crop_parent_indices = np.zeros((T, max_num_cells), dtype=np.int32)


    # Make active_mask from t_cell_tracks
    for t in range(T):
        t_cell_frame = t_cell_tracks[t]
        frame_ids = np.unique(t_cell_frame[t_cell_frame > 0])
        for cid in frame_ids:
            crop_active_mask[t, id_to_col[int(cid)]] = True

    # 4. Build is_new_root_mask, parent_indices (keep is_division_mask as all False)
    for cid, col in id_to_col.items():
        active_frames = np.where(crop_active_mask[:, col])[0]
        if len(active_frames) == 0:
            print(f"Warning: Cell ID {cid} (col {col}) is never active in active_mask. Skipping.")
            continue

        first_frame = active_frames[0]

        # This cell is a root (appeared spontaneously or is the initial cell)
        crop_is_new_root_mask[first_frame, col] = True
        crop_parent_indices[first_frame, col] = col  # self

        # For all subsequent active frames, parent = self (cell continues)
        for t in active_frames[1:]:
            crop_parent_indices[t, col] = col

    # Append crop masks to overall masks
    active_mask_list.append(crop_active_mask)
    is_division_mask_list.append(crop_is_division_mask)
    is_new_root_mask_list.append(crop_is_new_root_mask)
    parent_indices_list.append(crop_parent_indices)


crop=B4_t50t100y200y350x750x900, T=50, num_cells=57
crop=B8_t50t100y200y350x750x900, T=50, num_cells=77
crop=E4_t50t100y200y350x750x900, T=50, num_cells=22
crop=B4_t250t300y200y350x750x900, T=50, num_cells=55
crop=B8_t250t300y200y350x750x900, T=50, num_cells=88
crop=E4_t250t300y200y350x750x900, T=50, num_cells=27


In [15]:
active_mask = np.array(active_mask_list)
is_division_mask = np.array(is_division_mask_list)
is_new_root_mask = np.array(is_new_root_mask_list)
parent_indices = np.array(parent_indices_list)

print(f"active_mask shape:       {active_mask.shape}")
print(f"is_division_mask shape:  {is_division_mask.shape}")
print(f"is_new_root_mask shape:  {is_new_root_mask.shape}")
print(f"parent_indices shape:    {parent_indices.shape}")

active_mask shape:       (6, 50, 88)
is_division_mask shape:  (6, 50, 88)
is_new_root_mask shape:  (6, 50, 88)
parent_indices shape:    (6, 50, 88)


In [16]:
data['active_mask'] = active_mask
data['is_division_mask'] = is_division_mask
data['is_new_root_mask'] = is_new_root_mask
data['parent_indices'] = parent_indices

### Calculate feature values

In [18]:
# calculate emissions feature statistics

# calculate t cell velocities
t_cell_velocities_per_frame = {well: compute_cell_velocities_per_frame_dict(t_cell_tracks_per_well[well], unit_per_frame=1) for well in t_cell_tracks_per_well.keys()}


Computing cell velocities: 100%|██████████████████████████████████████████████████████████████████████████████████████| 49/49 [00:00<00:00, 1258.29it/s]


In [19]:
# calculate type-specific interactions

cancer_type_specific_contacts_per_frame, t_cell_type_specific_contacts_per_frame = {}, {}
cancer_type_specific_neighbors_per_frame, t_cell_type_specific_neighbors_per_frame = {}, {}

for well in sam3_tracks_per_well.keys():
    t_cell_tracks = t_cell_tracks_per_well[well]
    cancer_tracks = cancer_tracks_per_well[well]

    well_cancer_contacts_per_frame, well_t_cell_contacts_per_frame = compute_cell_cell_contact_dict(t_cell_tracks, cancer_tracks)
    well_cancer_neighbors_per_frame, well_t_cell_neighbors_per_frame = compute_cell_cell_neighbor_dict(t_cell_tracks, cancer_tracks)

    cancer_type_specific_contacts_per_frame[well] = well_cancer_contacts_per_frame
    t_cell_type_specific_contacts_per_frame[well] = well_t_cell_contacts_per_frame

    cancer_type_specific_neighbors_per_frame[well] = well_cancer_neighbors_per_frame
    t_cell_type_specific_neighbors_per_frame[well] = well_t_cell_neighbors_per_frame

Computing cell-cell contact dataframe...


Processing frame-by-frame contact output: 100%|████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 169.67it/s]


Computing cell-cell neighbor dataframe...


Processing frame-by-frame neighbor output: 100%|███████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 226.85it/s]


Computing cell-cell contact dataframe...


...

Processing frame-by-frame neighbor output: 100%|███████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 292.57it/s]


Computing cell-cell contact dataframe...


Processing frame-by-frame contact output: 100%|████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 177.33it/s]


Computing cell-cell neighbor dataframe...


Processing frame-by-frame neighbor output: 100%|███████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 240.72it/s]


In [20]:
# calculate type-agnostic interactions

cancer_all_contacts_per_frame, t_cell_all_contacts_per_frame = {}, {}
cancer_all_neighbors_per_frame, t_cell_all_neighbors_per_frame = {}, {}

for well in sam3_tracks_per_well.keys():
    t_cell_tracks = t_cell_tracks_per_well[well]
    cancer_tracks = cancer_tracks_per_well[well]

    well_cancer_contacts_per_frame, well_t_cell_contacts_per_frame = compute_all_cell_cell_contact_dict(t_cell_tracks, cancer_tracks)
    well_cancer_neighbors_per_frame, well_t_cell_neighbors_per_frame = compute_all_cell_cell_neighbor_dict(t_cell_tracks, cancer_tracks)

    cancer_all_contacts_per_frame[well] = well_cancer_contacts_per_frame
    t_cell_all_contacts_per_frame[well] = well_t_cell_contacts_per_frame

    cancer_all_neighbors_per_frame[well] = well_cancer_neighbors_per_frame
    t_cell_all_neighbors_per_frame[well] = well_t_cell_neighbors_per_frame

Computing cell-cell contact dataframe...


Processing contact dict: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 138.72it/s]


Computing cell-cell neighbor dataframe...


Processing neighbor dict: 100%|████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 151.18it/s]


Computing cell-cell contact dataframe...


...

Processing neighbor dict: 100%|████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 125.91it/s]


Computing cell-cell contact dataframe...


Processing contact dict: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 123.70it/s]


Computing cell-cell neighbor dataframe...


Processing neighbor dict: 100%|████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 134.33it/s]


### Create emissions array

In [ ]:
# emissions_array = shape: (# wells/batches, num_frames, num_t_cells that is max across all batches, num_emission_features)

emissions_array_list = []

for crop in crop_ids:
    t_cell_tracks = t_cell_tracks_per_well[crop]
    t_cell_velocities = t_cell_velocities_per_frame[crop]
    t_cell_type_specific_neighbors = t_cell_type_specific_neighbors_per_frame[crop]
    t_cell_all_neighbors = t_cell_all_neighbors_per_frame[crop]

    T = t_cell_tracks.shape[0]
    t_cell_ids = np.unique(t_cell_tracks[t_cell_tracks > 0])
    t_cell_ids.sort()
    id_to_column_index = {cell_id: index for index, cell_id in enumerate(t_cell_ids)}
    
    print(f"crop={crop}, T={T}, num_cells={len(t_cell_ids)}")

    crop_emissions = np.zeros((T, max_num_cells, 3)) # 3 features: velocity, type-specific neighbors, all neighbors

    for t in range(T):
        frame = t_cell_tracks[t]
        for cell_id in np.unique(frame[frame > 0]):
            column_index = id_to_column_index[cell_id]

            if t > 0:
                if cell_id in t_cell_velocities[t]:
                    velocity = t_cell_velocities[t][cell_id]
                    crop_emissions[t, column_index, 0] = velocity

            if cell_id in t_cell_all_neighbors[t]:
                all_neighbors = t_cell_all_neighbors[t][cell_id]
                cancer_neighbors_list = t_cell_type_specific_neighbors[t].get(cell_id, [0])
                cancer_neighbors = cancer_neighbors_list[0]

                t_cell_neighbors = all_neighbors - cancer_neighbors

                crop_emissions[t, column_index, 1] = cancer_neighbors
                crop_emissions[t, column_index, 2] = t_cell_neighbors

    emissions_array_list.append(crop_emissions)
    print(f"emissions_array shape after processing {crop}: {crop_emissions.shape}")

crop=B4_t50t100y200y350x750x900, T=50, num_cells=57
emissions_array shape after processing B4_t50t100y200y350x750x900: (50, 88, 3)
crop=B8_t50t100y200y350x750x900, T=50, num_cells=77
emissions_array shape after processing B8_t50t100y200y350x750x900: (50, 88, 3)
crop=E4_t50t100y200y350x750x900, T=50, num_cells=22
emissions_array shape after processing E4_t50t100y200y350x750x900: (50, 88, 3)
crop=B4_t250t300y200y350x750x900, T=50, num_cells=55
emissions_array shape after processing B4_t250t300y200y350x750x900: (50, 88, 3)
crop=B8_t250t300y200y350x750x900, T=50, num_cells=88
emissions_array shape after processing B8_t250t300y200y350x750x900: (50, 88, 3)
crop=E4_t250t300y200y350x750x900, T=50, num_cells=27
emissions_array shape after processing E4_t250t300y200y350x750x900: (50, 88, 3)


In [24]:
emissions_array = np.array(emissions_array_list)
print(f"Final emissions_array shape: {emissions_array.shape}")

Final emissions_array shape: (6, 50, 88, 3)


### Generate crop-specific array col idx mapping

In [25]:
all_crop_cell_ids = []

for crop in crop_ids:
    t_cell_tracks = t_cell_tracks_per_well[crop]
    t_cell_ids = np.unique(t_cell_tracks[t_cell_tracks > 0])
    t_cell_ids.sort()

    filler_cell_ids = [f"{crop}_filler_{i}" for i in range(max_num_cells - len(t_cell_ids))]

    crop_t_cell_ids = [f"{crop}_{cell_id}" for cell_id in t_cell_ids]

    # add filler bc need to have consistent cell numbers
    crop_t_cell_ids.extend(filler_cell_ids)

    all_crop_cell_ids.extend(crop_t_cell_ids)

id_to_col = {cid: i for i, cid in enumerate(all_crop_cell_ids)}

In [28]:
len(id_to_col.keys())

528

### save data

In [30]:
output_dir = "/gladstone/engelhardt/lab/jutran/lci/treeHMM/notebooks/data/all_gt_crops_pretrained_SAM3/"

with open(output_dir + "batched_data.pkl", "wb") as f:
    pickle.dump(data, f)

with open(output_dir + "batched_crop_and_cell_id_to_col.pkl", "wb") as f:
    pickle.dump(id_to_col, f)

np.save(output_dir + "batched_emissions_array.npy", emissions_array)